# Setup Vector and Graph DB

Pyeed consists of a vector database (Milvus) and a graph database (Neo4j).
The vector database is used to store vector embeddings of protein sequences to search for similar sequences.

The graph database stores the protein sequences and annotated information in a semantic graph.

Both datases can be setup via Docker Compose.

```bash
version: "3.5"

services:
  # ======================
  # Neo4j graph database
  # ======================
  neo4j:
    image: neo4j:5.26.14-community
    container_name: neo4j
    restart: unless-stopped

    ports:
      - "127.0.0.1:7474:7474"   # Neo4j Browser / HTTP
      - "127.0.0.1:7687:7687"   # Bolt

    environment:
      NEO4J_AUTH: "neo4j/12345678"

      # memory tuning
      NEO4J_server_memory_heap_initial__size: 16g
      NEO4J_server_memory_heap_max__size: 16g
      NEO4J_server_memory_pagecache_size: 64g

      # plugins
      NEO4J_dbms_security_procedures_unrestricted: "apoc.*"
      NEO4J_PLUGINS: '["apoc", "graph-data-science"]'

    volumes:
      - /home/mha/dbs/neo4j/proteingraph/data:/data
      - /home/mha/dbs/neo4j/proteingraph/logs:/logs
      - /home/mha/dbs/neo4j/proteingraph/import:/import
      - /home/mha/dbs/neo4j/proteingraph/plugins:/plugins

    networks:
      - backend

  # ======================
  # Milvus dependencies
  # ======================

  etcd:
    container_name: milvus-etcd
    image: quay.io/coreos/etcd:v3.5.18
    restart: unless-stopped
    environment:
      ETCD_AUTO_COMPACTION_MODE: revision
      ETCD_AUTO_COMPACTION_RETENTION: "1000"
      ETCD_QUOTA_BACKEND_BYTES: "4294967296"
      ETCD_SNAPSHOT_COUNT: "50000"
    command: >
      etcd
      -advertise-client-urls=http://etcd:2379
      -listen-client-urls=http://0.0.0.0:2379
      --data-dir=/etcd
    volumes:
      - /home/mha/dbs/milvus/etcd:/etcd
    healthcheck:
      test: ["CMD", "etcdctl", "endpoint", "health"]
      interval: 30s
      timeout: 20s
      retries: 3
    networks:
      - backend

  minio:
    container_name: milvus-minio
    image: minio/minio:RELEASE.2024-12-18T13-15-44Z
    restart: unless-stopped
    environment:
      MINIO_ACCESS_KEY: minioadmin
      MINIO_SECRET_KEY: minioadmin
    command: ["server", "/minio_data", "--console-address", ":9001"]
    # bind only to localhost so it's not public
    ports:
      - "127.0.0.1:9000:9000"   # S3 API
      - "127.0.0.1:9001:9001"   # MinIO console (optional)
    volumes:
      - /home/mha/dbs/milvus/minio:/minio_data
    healthcheck:
      test: ["CMD", "curl", "-f", "http://localhost:9000/minio/health/live"]
      interval: 30s
      timeout: 20s
      retries: 3
    networks:
      - backend

  # ======================
  # Milvus standalone
  # ======================
  milvus:
    container_name: milvus-standalone
    image: milvusdb/milvus:v2.6.4
    restart: unless-stopped
    command: ["milvus", "run", "standalone"]
    security_opt:
      - seccomp:unconfined

    environment:
      ETCD_ENDPOINTS: etcd:2379
      MINIO_ADDRESS: minio:9000
      MQ_TYPE: woodpecker

    # bind Milvus API to localhost only
    ports:
      - "127.0.0.1:19530:19530" # gRPC
      - "127.0.0.1:9091:9091"   # REST / metrics / healthz

    volumes:
      - /home/mha/dbs/milvus/data:/var/lib/milvus

    healthcheck:
      test: ["CMD", "curl", "-f", "http://localhost:9091/healthz"]
      interval: 30s
      start_period: 90s
      timeout: 20s
      retries: 3

    depends_on:
      - etcd
      - minio
    networks:
      - backend

networks:
  backend:
    driver: bridge
```

Execute the following command to start the services:

```bash
docker compose up -d
```
